In [0]:
# 1. Limpieza del entorno (Elimina tabla y carpetas corruptas del primer intento)
spark.sql("DROP TABLE IF EXISTS certificacion.lab01.bronze_felipe")
dbutils.fs.rm("/Volumes/certificacion/default/datasetlab1/_checkpoints/bronze_felipe/", True)
dbutils.fs.rm("/Volumes/certificacion/default/datasetlab1/_esquemas/bronze_felipe/", True)

# 2. Definición de rutas limpias (Nombres oficiales)
origen_volumen = "/Volumes/certificacion/default/datasetlab1/"
ruta_checkpoint = "/Volumes/certificacion/default/datasetlab1/_checkpoints/bronze_felipe/"
ruta_esquema = "/Volumes/certificacion/default/datasetlab1/_esquemas/bronze_felipe/"

# 3. Configuración de Auto Loader (Con multilínea, escape y filtro)
df_bronze = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("multiLine", "true")
    .option("escape", "\"")
    .option("cloudFiles.schemaLocation", ruta_esquema)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("pathGlobFilter", "instagram_reels_clean.csv")
    .load(origen_volumen)
)

# 4. Escritura oficial a la capa Bronze
(df_bronze.writeStream
    .format("delta")
    .option("checkpointLocation", ruta_checkpoint)
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable("certificacion.lab01.bronze_felipe")
)